# Reading FITS files

This notebook is used to explore the FITS files located in the `../data` directory.

In [ ]:
from astropy.io import fits
import glob
import os
import pandas as pd
import sys
import getpass
from multiprocessing import BoundedSemaphore, Lock

In [ ]:
# Add hsc_to_lsst to python path to import pipeline
sys.path.append(os.path.abspath("HSC_to_LSST"))
try:
    from hsc_to_lsst.pipeline import query_and_write
    print("Successfully imported query_and_write from hsc_to_lsst.pipeline")
except ImportError as e:
    print(f"Error importing hsc_to_lsst: {e}")
    print("Make sure the HSC_to_LSST directory is in the current directory or python path.")

In [ ]:
# List all fits files in ../data
data_dir = "../data"
fits_files = glob.glob(os.path.join(data_dir, "*.fits"))
print(f"Found {len(fits_files)} FITS files in {data_dir}:")
for f in fits_files:
    print(f)

In [ ]:
def explore_fits(file_path):
    print(f"\n{'='*80}\nExploring: {file_path}\n{'='*80}")
    try:
        with fits.open(file_path) as hdul:
            hdul.info()
            
            for i, hdu in enumerate(hdul):
                print(f"\n--- HDU {i} ({hdu.name}) ---")
                display(hdu.header)
                
                if isinstance(hdu, fits.BinTableHDU) or isinstance(hdu, fits.TableHDU):
                    print(f"Table columns: {hdu.columns.names}")
                    # Convert to pandas data frame for nicer display of first few rows
                    try:
                        from astropy.table import Table
                        df = Table(hdu.data).to_pandas()
                        display(df.head())
                        print(f"Table shape: {df.shape}")
                    except Exception as e:
                        print(f"Could not convert to DataFrame: {e}")
                elif isinstance(hdu, fits.ImageHDU) or isinstance(hdu, fits.PrimaryHDU):
                     if hdu.data is not None:
                        print(f"Image data shape: {hdu.data.shape}")
                        print(f"Min: {hdu.data.min()}, Max: {hdu.data.max()}")
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

In [ ]:
# Load specific files into DataFrames
wide_path = os.path.join(data_dir, "sugohi_lenses_AB_wide.fits")
deep_path = os.path.join(data_dir, "sugohi_lenses_AB_deep.fits")

def load_fits_table(file_path):
    try:
        with fits.open(file_path) as hdul:
            # Assuming the table is in the first extension (index 1) which is standard for FITS tables
            # Adjust if necessary based on hdul.info()
            if len(hdul) > 1 and (isinstance(hdul[1], fits.BinTableHDU) or isinstance(hdul[1], fits.TableHDU)):
                from astropy.table import Table
                # Use astropy.table.Table for robust conversion to Pandas, handling endianness
                df = Table(hdul[1].data).to_pandas()
                return df
            else:
                print(f"No table found in extension 1 of {file_path}")
                return None
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

# Must import numpy as it's used in buffer handling implicit or explicit
import numpy as np

wide = load_fits_table(wide_path)
if wide is not None:
    print(f"Loaded 'wide' DataFrame with shape: {wide.shape}")

deep = load_fits_table(deep_path)
if deep is not None:
    print(f"Loaded 'deep' DataFrame with shape: {deep.shape}")

# Define columns to display
columns_to_show = ['ra', 'dec', 'type', 'discovery', 'grade', 'Reference', 'object_id']

if wide is not None:
    print("\n--- Wide DataFrame (Selected Columns) ---")
    try:
        display(wide[columns_to_show].head())
    except KeyError as e:
        print(f"Error selecting columns in 'wide': {e}. Available columns: {wide.columns.tolist()}")

if deep is not None:
    print("\n--- Deep DataFrame (Selected Columns) ---")
    try:
        display(deep[columns_to_show].head())
    except KeyError as e:
        print(f"Error selecting columns in 'deep': {e}. Available columns: {deep.columns.tolist()}")

## Query HSC Database

The following cells allow you to query the HSC database for the objects in the DataFrames. You will need to provide your HSC SSP username and password.

In [ ]:
# Input HSC credentials securely
print("Please enter your HSC SSP credentials:")
hsc_username = getpass.getpass("HSC Username: ")
hsc_password = getpass.getpass("HSC Password: ")

In [ ]:
def query_hsc_for_dataframe(df, output_base_dir, hsc_release="pdr3_wide", limit=5):
    """
    Query HSC for objects in the DataFrame.
    
    Parameters:
        df: pandas DataFrame containing 'ra' and 'dec' columns
        output_base_dir: Directory where output images will be saved
        hsc_release: HSC data release to query (e.g., 'pdr3_wide', 'pdr3_dud')
        limit: Number of objects to query (set to None for all)
    """
    
    if df is None:
        print("DataFrame is None, skipping query.")
        return

    os.makedirs(output_base_dir, exist_ok=True)
    
    # Semaphore to limit concurrent connections (as used in query_hsc script)
    semaphore = BoundedSemaphore(value=4)
    log_lock = Lock()
    log_file = os.path.join(output_base_dir, "query_log.txt")
    
    count = 0
    for index, row in df.iterrows():
        if limit is not None and count >= limit:
            break
            
        object_id = row.get('object_id', index)
        ra = row['ra']
        dec = row['dec']
        
        # Create a unique output directory/filename for each object if needed
        # The query_and_write function creates {out_filename}_img_{band}.fits
        out_filename = os.path.join(output_base_dir, f"{object_id}")
        
        print(f"Querying object {object_id} at RA={ra}, Dec={dec}...")
        
        success = query_and_write(
            out_filename,
            ra,
            dec,
            semaphore,
            username=hsc_username,
            password=hsc_password,
            query_timeuot=60,
            hsc_size_arcsec=20,
            field=hsc_release,
            bands='grizy',
            log_file=log_file,
            log_lock=log_lock,
            log_prefix=f"{object_id}: ",
            require_psf=True
        )
        
        if success:
            print(f"  -> Success! Saved to {out_filename}_*")
        else:
            print(f"  -> Failed. Check {log_file} for details.")
            
        count += 1

# Example usage: Query first 2 objects from 'wide' DataFrame
if wide is not None:
    print("\n--- Querying Wide Data ---")
    query_hsc_for_dataframe(wide, "../data/hsc_query_output/wide", hsc_release="pdr3_wide", limit=None)

# Example usage: Query first 2 objects from 'deep' DataFrame
if deep is not None:
    print("\n--- Querying Deep Data ---")
    # Use appropriate deep release, e.g., pdr3_dud (Deep/UltraDeep)
    query_hsc_for_dataframe(deep, "../data/hsc_query_output/deep", hsc_release="pdr3_dud", limit=None)

## Display Retrieved Images

The following cell displays the images retrieved from the HSC database.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import glob
import os
from astropy.io import fits

def display_retrieved_images(output_dir):
    print(f"Displaying images from {output_dir}")
    # Get list of object IDs from filenames
    files = glob.glob(os.path.join(output_dir, "*_img_*.fits"))
    if not files:
        print("No images found.")
        return

    # Group by object ID
    objects = {}
    for f in files:
        basename = os.path.basename(f)
        parts = basename.split('_')
        # format: {object_id}_img_{band}.fits
        if len(parts) >= 3:
            obj_id = parts[0]
            band = parts[2].split('.')[0]
            
            if obj_id not in objects:
                objects[obj_id] = {}
            objects[obj_id][band] = f

    # Display images
    for obj_id, bands in objects.items():
        print(f"Object: {obj_id}")
        sorted_bands = sorted(bands.keys())
        num_bands = len(sorted_bands)
        
        if num_bands == 0:
            continue
            
        # Create figure
        fig, axes = plt.subplots(1, num_bands, figsize=(4*num_bands, 4))
        if num_bands == 1:
            axes = [axes]
        
        for ax, band in zip(axes, sorted_bands):
            fpath = bands[band]
            try:
                with fits.open(fpath) as hdul:
                    # Assuming image data is in primary HDU or first extension
                    img_data = hdul[0].data if hdul[0].data is not None else hdul[1].data
                    
                    # Simple normalization for display
                    valid_data = img_data[~np.isnan(img_data)]
                    if len(valid_data) > 0:
                        vmin = np.percentile(valid_data, 1)
                        vmax = np.percentile(valid_data, 99)
                    else:
                        vmin, vmax = 0, 1
                    
                    ax.imshow(img_data, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
                    ax.set_title(f"Band: {band}")
                    ax.axis('off')
            except Exception as e:
                print(f"Error reading {fpath}: {e}")
                
        plt.tight_layout()
        plt.show()

# Display images for Wide and Deep
output_dir_wide = "../data/hsc_query_output/wide"
if os.path.exists(output_dir_wide):
    print("\n--- Wide Images ---")
    display_retrieved_images(output_dir_wide)

output_dir_deep = "../data/hsc_query_output/deep"
if os.path.exists(output_dir_deep):
    print("\n--- Deep Images ---")
    display_retrieved_images(output_dir_deep)

## Apply LSST Degradation

This section applies the `read_degrade_hsc` function to the retrieved HSC images to simulate LSST-like images.
It then displays the original HSC images side-by-side with the degraded LSST images.

### Note on Image Sizes and PSF

- **Image Sizes**: The HSC images are approximately 20 arcseconds on a side (119x119 pixels at 0.168"/pix). The LSST images are 8.2 arcseconds on a side (41x41 pixels at 0.2"/pix). In the comparison plots below, the **HSC images are zoomed in** to match the field of view of the LSST images (8.2" x 8.2") to allow for a direct comparison of features at the same angular size.
- **PSF Model**: The LSST PSF is modeled as a Gaussian kernel. The FWHM (Full Width at Half Maximum) of this Gaussian is sampled from LSST DP0.2 simulations, which provide realistic seeing conditions expected for the LSST survey.

In [ ]:
from hsc_to_lsst.pipeline import read_degrade_write
from hsc_to_lsst.lsst_props import dp0_gmm_sampler
from multiprocessing import Lock
import os
import glob
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits

def perform_degradation(input_dir, output_dir, coadd_years=1):
    print(f"Degrading images from {input_dir} to {output_dir}...")
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize sampler
    dp0_sampler = dp0_gmm_sampler(str(coadd_years))
    log_lock = Lock()
    log_file = os.path.join(output_dir, "degrade.log")

    # Find objects
    files = glob.glob(os.path.join(input_dir, "*_img_*.fits"))
    input_ids = sorted(list(set([os.path.basename(f).split("_")[0] for f in files])))

    if not input_ids:
        print("No input images found.")
        return

    for idx in input_ids:
        print(f"Processing object {idx}...")
        in_filename = os.path.join(input_dir, idx)
        out_filename = os.path.join(output_dir, idx)
        
        try:
            success, mag_change = read_degrade_write(
                in_filename,
                out_filename,
                dp0_sampler,
                zp_rms_frac_thresh=0.3,
                zp_max_mag_change=2.0,
                lsst_size_pix=41,
                bands='grizy',
                log_file=log_file,
                log_lock=log_lock,
                log_prefix=f"{idx}: "
            )
            if success:
                print(f"  -> Success! Mag change: {mag_change:.3f}")
            else:
                 print(f"  -> Failed. Check {log_file}")
        except Exception as e:
            print(f"  -> Error: {e}")

def display_comparison(hsc_dir, lsst_dir):
    print(f"Comparing images from {hsc_dir} (HSC) and {lsst_dir} (LSST)")
    print(f"Current working directory: {os.getcwd()}")
    print(f"Absolute HSC path: {os.path.abspath(hsc_dir)}")
    print(f"Absolute LSST path: {os.path.abspath(lsst_dir)}")
    
    # Find LSST files
    files = glob.glob(os.path.join(lsst_dir, "*.fits"))
    print(f"Found {len(files)} files in LSST dir: {files[:5]}...")
    
    objects = {}
    for f in files:
        basename = os.path.basename(f)
        parts = basename.rsplit('_', 1)
        if len(parts) == 2:
             obj_id = parts[0]
             band_ext = parts[1]
             if '.' in band_ext:
                 band = band_ext.split('.')[0]
                 if obj_id not in objects:
                     objects[obj_id] = {'lsst': {}, 'hsc': {}}
                 objects[obj_id]['lsst'][band] = f
                 
                 # HSC format: {id}_img_{band}.fits
                 hsc_path = os.path.join(hsc_dir, f"{obj_id}_img_{band}.fits")
                 if os.path.exists(hsc_path):
                     objects[obj_id]['hsc'][band] = hsc_path
                 else:
                     print(f"Warning: Missing HSC file for {obj_id} band {band} at {hsc_path}")

    if not objects:
        print("No matched objects found to display.")
        return

    for obj_id, data in objects.items():
        print(f"Displaying Object: {obj_id}")
        bands = sorted(data['lsst'].keys())
        if not bands:
            continue
            
        num_bands = len(bands)
        fig, axes = plt.subplots(2, num_bands, figsize=(4*num_bands, 8))
        if num_bands == 1:
            axes = np.array([axes]).T
            
        for i, band in enumerate(bands):
            # Setup HSC
            if band in data['hsc']:
                with fits.open(data['hsc'][band]) as hdul:
                    img = hdul[0].data if hdul[0].data is not None else hdul[1].data
                    
                    # HSC pixel scale = 0.168 arcsec/pix
                    hsc_pix_scale = 0.168
                    ny, nx = img.shape
                    hsc_extent = [-nx*hsc_pix_scale/2, nx*hsc_pix_scale/2, -ny*hsc_pix_scale/2, ny*hsc_pix_scale/2]
                    
                    vmin, vmax = np.percentile(img[~np.isnan(img)], [1, 99]) if np.any(~np.isnan(img)) else (0, 1)
                    axes[0, i].imshow(img, origin='lower', cmap='gray', vmin=vmin, vmax=vmax, extent=hsc_extent)
                    axes[0, i].set_title(f"HSC {band} (Zoomed)")
            else:
                axes[0, i].text(0.5, 0.5, "Missing", ha='center')
                axes[0, i].axis('off')

            # Setup LSST
            with fits.open(data['lsst'][band]) as hdul:
                 img = hdul[0].data
                 
                 # LSST pixel scale = 0.2 arcsec/pix
                 lsst_pix_scale = 0.2
                 ny, nx = img.shape
                 lsst_extent = [-nx*lsst_pix_scale/2, nx*lsst_pix_scale/2, -ny*lsst_pix_scale/2, ny*lsst_pix_scale/2]

                 vmin, vmax = np.percentile(img[~np.isnan(img)], [1, 99]) if np.any(~np.isnan(img)) else (0, 1)
                 axes[1, i].imshow(img, origin='lower', cmap='gray', vmin=vmin, vmax=vmax, extent=lsst_extent)
                 axes[1, i].set_title(f"LSST {band}")
                 
                 # Synch axes limits of HSC to LSST
                 axes[0, i].set_xlim(lsst_extent[0], lsst_extent[1])
                 axes[0, i].set_ylim(lsst_extent[2], lsst_extent[3])
                 
        plt.tight_layout()
        
        # Save figure
        save_path = os.path.join(lsst_dir, f"{obj_id}_comparison.png")
        plt.savefig(save_path)
        print(f"Saved comparison plot to {save_path}")
        plt.show()

# Run degradation and display for Wide
if os.path.exists(output_dir_wide):
    degraded_dir_wide = output_dir_wide + "_degraded"
    print("\n--- Degrading Wide Data ---")
    perform_degradation(output_dir_wide, degraded_dir_wide)
    display_comparison(output_dir_wide, degraded_dir_wide)

# Run degradation and display for Deep
if os.path.exists(output_dir_deep):
    degraded_dir_deep = output_dir_deep + "_degraded"
    print("\n--- Degrading Deep Data ---")
    perform_degradation(output_dir_deep, degraded_dir_deep)
    display_comparison(output_dir_deep, degraded_dir_deep)